In [ ]:
import pandas as pd
import numpy as np

file_path = "/content/과학기술정보통신부 우정사업본부_전국 우편번호(배달지 기준)별 택배물량_20231231.csv"
df = pd.read_csv(file_path, encoding="cp949")


print("---[1] 원본 데이터 크기 및 결측치 점검")
print(f"행, 열 크기: {df.shape}")
print(f"결측치 총합: {df.isnull().sum().sum()}개")

# 전처리 1
df['우편번호 5자리'] = df["우편번호"].astype(str).str.zfill(5) # 우편번호 5자리 (1000 -> 01000)

# 전처리 2 , 우편번호 앞 2자리로 시/도 구분
def get_sido(zip_code):
  prefix = int(zip_code[:2])

  if 1 <= prefix <= 9:
        return "서울특별시"

  elif 10 <= prefix <= 20:
        return "경기도"

  elif 21 <= prefix <= 23:
        return "인천광역시"

  elif 24 <= prefix <= 26:
        return "강원특별자치도"

  elif 27 <= prefix <= 29:
        return "충청북도"

  elif prefix == 30:
        return "세종특별자치시"

  elif 31 <= prefix <= 33:
        return "충청남도"

  elif 34 <= prefix <= 35:
        return "대전광역시"

  elif 36 <= prefix <= 40:
        return "경상북도"

  elif 41 <= prefix <= 43:
        return "대구광역시"

  elif 44 <= prefix <= 45:
        return "울산광역시"

  elif 46 <= prefix <= 49:
        return "부산광역시"

  elif 50 <= prefix <= 53:
        return "경상남도"

  elif 54 <= prefix <= 56:
        return "전북특별자치도"

  elif 57 <= prefix <= 60:
        return "전라남도"

  elif 61 <= prefix <= 62:
        return "광주광역시"

  elif prefix == 63:
        return "제주특별자치도"
  return "기타"

df["시도"] = df["우편번호 5자리"].apply(get_sido)

# 전처리 3, 연간 총 물동량 계산
month_cols = [f"{i}월" for i in range(1,13)]
df["연간총물동량"] = df[month_cols].sum(axis=1)

# 전처리 4
# Melt를 사용하여 Wide 양식을 Long 양식으로 변환
df_long = df.melt(
    id_vars=['우편번호 5자리', '시도'],
    value_vars=month_cols,
    var_name='월',  # 컬럼명이 될 이름
    value_name='물동량',  # 실제 수치 데이터 컬럼명
)

# 전처리 5 설날, 추석, 피크 시즌 파생변수 생성
df_long["월 숫자"] = df_long["월"].str.replace("월","").astype(int)
df_long["피크시즌"] = df_long["월 숫자"].isin([1,2,9,10,12]).astype(int)

print("\n 전처리 완료된 데이터 요약 ")
print(f"전처리 후 총 행 수: {len(df_long):,}개")
print(df_long.head(10))

print("=== 전국 시/도별 연간 총 물동량 요약 ===")
sido_summary = (df.groupby("시도")["연간총물동량"].sum().reset_index().sort_values(by="연간총물동량", ascending=False))
print(sido_summary.to_string(index=False))

# 전처리 완료된 파일 저장
df_long.to_csv("preprocessed_parcel_data.csv", index=False, encoding="utf-8-sig")
df.to_csv("summary_by_zipcode.csv", index=False, encoding="utf-8-sig")

print("\n'preprocessed_parcel_data.csv' 파일로 저장 완료!")

---[1] 원본 데이터 크기 및 결측치 점검
행, 열 크기: (34502, 13)
결측치 총합: 0개

 전처리 완료된 데이터 요약 
전처리 후 총 행 수: 414,024개
  우편번호 5자리     시도   월  물동량  월 숫자  피크시즌
0    01000  서울특별시  1월   98     1     1
1    01001  서울특별시  1월  253     1     1
2    01002  서울특별시  1월  723     1     1
3    01003  서울특별시  1월  394     1     1
4    01004  서울특별시  1월  281     1     1
5    01005  서울특별시  1월  229     1     1
6    01006  서울특별시  1월  469     1     1
7    01007  서울특별시  1월   94     1     1
8    01008  서울특별시  1월  138     1     1
9    01009  서울특별시  1월  195     1     1
=== 전국 시/도별 연간 총 물동량 요약 ===
     시도   연간총물동량
    경기도 61755464
  서울특별시 47995462
  부산광역시 13018899
  인천광역시 12508410
   경상남도 12194933
   경상북도  9243728
  대구광역시  9173116
   충청남도  8673311
전북특별자치도  6629350
   충청북도  6554968
   전라남도  6487730
강원특별자치도  6362653
  광주광역시  6330874
  대전광역시  6052338
  울산광역시  4322295
제주특별자치도  2847334
세종특별자치시  2015571

'preprocessed_parcel_data.csv' 파일로 저장 완료!
